# NLP-Based Resume–Job Alignment — Week 1

## Research question and hypothesis
Does combining semantic similarity with lexical matching improve identification of relevant resume–job qualifications compared with lexical matching alone?

My working hypothesis is that semantic information may help with differently worded qualifications. This notebook establishes synthetic inputs and initial lexical rankings only. Semantic matching and comparative ranking metrics remain planned.

## Execution instructions
Open this notebook from the repository root or its `src` folder and select a Python kernel. Install the packages listed in `src/requirements.txt` if needed. Restart the kernel, run all cells in order, and save the notebook. Outputs are written to `results/week_1/`.


In [1]:
from pathlib import Path
import json, platform
import numpy as np
import pandas as pd
import sklearn
def display(frame):
    print(frame.to_string(index=True))
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().resolve()
if ROOT.name == "src":
    ROOT = ROOT.parent
assert (ROOT / "src" / "week1_resume_job_foundation.ipynb").exists(), "Open from the extracted project root or src folder."
OUT = ROOT / "results" / "week_1"
OUT.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Output folder:", OUT)
print("Python:", platform.python_version())

Project root: /home/user/tejaswi
Output folder: /home/user/tejaswi/results/week_1
Python: 3.12.8


## Synthetic inputs and relevance rubric
I construct fictional examples for data analysis, web development, and IT support. Each family includes direct wording, paraphrased wording, and partial qualifications. No personal resumes are used.

Grades are assigned before scores: 2 = substantial intended task/tool coverage, 1 = partial coverage, 0 = no relevant evidence in this toy scenario. These are author-defined scenario judgments, not independent annotations or hiring-suitability labels. Job-family grouping is deliberately simplified. Negative wording is retained to expose a limitation of word-based similarity.

In [2]:
jobs = pd.DataFrame([
("J1", "Data analyst", "Analyze data using Python and SQL. Create dashboards and communicate findings."),
("J2", "Web developer", "Build web interfaces using JavaScript HTML and CSS. Test responsive websites."),
("J3", "IT support", "Troubleshoot Windows computers and network connections. Resolve help desk tickets.")
], columns=["job_id", "title", "text"])
resumes = pd.DataFrame([
("R1", "Python and SQL data analysis. Created dashboards and presented findings."),
("R2", "Queried relational databases, analyzed datasets with pandas, and built visual reports for stakeholders."),
("R3", "Prepared spreadsheet reports and charts. No programming or database experience."),
("R4", "Built responsive websites with JavaScript HTML and CSS. Tested web interfaces."),
("R5", "Developed browser user interfaces with JS, styled pages with cascading style sheets, and checked layouts on phones."),
("R6", "Designed website mockups and page layouts. No coding experience."),
("R7", "Resolved help desk tickets, troubleshot Windows computers and network connections."),
("R8", "Diagnosed Microsoft desktop operating system faults, restored connectivity, and handled user service requests."),
("R9", "Assisted users with passwords and application installation. No network troubleshooting experience.")
], columns=["resume_id", "text"])
grades = {"J1":[2,2,1,0,0,0,0,0,0], "J2":[0,0,0,2,2,1,0,0,0], "J3":[0,0,0,0,0,0,2,2,1]}
labels = pd.DataFrame([(j,r,g) for j,gs in grades.items() for r,g in zip(resumes.resume_id,gs)], columns=["job_id","resume_id","relevance"])
display(jobs)
display(resumes)
display(labels.groupby(["job_id","relevance"]).size().unstack(fill_value=0))

  job_id          title                                                                                text
0     J1   Data analyst      Analyze data using Python and SQL. Create dashboards and communicate findings.
1     J2  Web developer       Build web interfaces using JavaScript HTML and CSS. Test responsive websites.
2     J3     IT support  Troubleshoot Windows computers and network connections. Resolve help desk tickets.
  resume_id                                                                                                                 text
0        R1                                             Python and SQL data analysis. Created dashboards and presented findings.
1        R2              Queried relational databases, analyzed datasets with pandas, and built visual reports for stakeholders.
2        R3                                      Prepared spreadsheet reports and charts. No programming or database experience.
3        R4                                       Bu

## Input validation
I check identifier integrity, text completeness, pair coverage, references, and grade values. Later checks test numerical behavior, not the validity of the scenario labels.

In [3]:
checks = {
"unique_job_ids": jobs.job_id.is_unique,
"unique_resume_ids": resumes.resume_id.is_unique,
"nonempty_text": bool(jobs.text.str.strip().ne("").all() and resumes.text.str.strip().ne("").all()),
"no_missing_values": not any(d.isna().any().any() for d in [jobs,resumes,labels]),
"unique_pairs": not labels.duplicated(["job_id","resume_id"]).any(),
"complete_pairs": len(labels) == len(jobs)*len(resumes),
"valid_references": set(labels.job_id)==set(jobs.job_id) and set(labels.resume_id)==set(resumes.resume_id),
"valid_grades": labels.relevance.isin([0,1,2]).all()
}
assert all(checks.values()), checks
print("Input checks passed:", len(checks))

Input checks passed: 8


## Initial lexical method
I fit unigram TF-IDF on the fixed resume collection only, transform the jobs using that vocabulary, and calculate cosine similarity. I retain the default tokenizer and no stop-word removal. Ties are resolved by resume ID. This is not a trained hiring model or a semantic-method comparison.

In [4]:
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1,1))
resume_vectors = vectorizer.fit_transform(resumes.text)
job_vectors = vectorizer.transform(jobs.text)
scores = cosine_similarity(job_vectors, resume_vectors)
checks["finite_bounded_scores"] = bool(np.isfinite(scores).all() and (scores >= -1e-12).all() and (scores <= 1+1e-12).all())
checks["self_similarity"] = bool(np.isclose(cosine_similarity(resume_vectors[0],resume_vectors[0])[0,0],1))
checks["unknown_text_zero"] = bool(np.allclose(cosine_similarity(vectorizer.transform(["zzzxxyyqq"]),resume_vectors),0))
validation = pd.DataFrame([(k,bool(v)) for k,v in checks.items()],columns=["check","passed"])
assert validation.passed.all(), validation
rankings = pd.DataFrame([(j,r,float(scores[i,k])) for i,j in enumerate(jobs.job_id) for k,r in enumerate(resumes.resume_id)],columns=["job_id","resume_id","lexical_score"])
rankings = rankings.merge(labels,on=["job_id","resume_id"],validate="one_to_one").sort_values(["job_id","lexical_score","resume_id"],ascending=[True,False,True])
rankings["rank"] = rankings.groupby("job_id").cumcount()+1
display(validation)
display(rankings.loc[rankings["rank"]<=3])

                    check  passed
0          unique_job_ids    True
1       unique_resume_ids    True
2           nonempty_text    True
3       no_missing_values    True
4            unique_pairs    True
5          complete_pairs    True
6        valid_references    True
7            valid_grades    True
8   finite_bounded_scores    True
9         self_similarity    True
10      unknown_text_zero    True
   job_id resume_id  lexical_score  relevance  rank
0      J1        R1       0.806630          2     1
5      J1        R6       0.047172          0     2
2      J1        R3       0.044101          1     3
12     J2        R4       0.873329          2     1
13     J2        R5       0.084883          2     2
9      J2        R1       0.038268          0     3
24     J3        R7       0.879927          2     1
26     J3        R9       0.113592          1     2
18     J3        R1       0.038268          0     3


## Save verifiable outputs
I save inputs separately from computed rankings and validation results. Re-running overwrites these generated files. Environment versions describe the kernel used for this run, not a full environment lock.

In [5]:
summary = {"jobs":len(jobs),"resumes":len(resumes),"pairs":len(labels),"vocabulary_terms":len(vectorizer.vocabulary_),"checks_passed":int(validation.passed.sum()),"semantic_matching_executed":False,"comparative_evaluation_executed":False}
for name,frame in [("jobs",jobs),("resumes",resumes),("relevance_labels",labels),("lexical_rankings",rankings),("validation_checks",validation)]:
    frame.to_csv(OUT / (name+".csv"), index=False)
(OUT/"execution_summary.json").write_text(json.dumps(summary,indent=2)+"\n", encoding="utf-8")
(OUT/"environment.json").write_text(json.dumps({"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,"scikit-learn":sklearn.__version__},indent=2)+"\n", encoding="utf-8")
print(json.dumps(summary,indent=2))
print("Saved files:", sorted(p.name for p in OUT.iterdir() if p.is_file()))

{
  "jobs": 3,
  "resumes": 9,
  "pairs": 27,
  "vocabulary_terms": 81,
  "checks_passed": 11,
  "semantic_matching_executed": false,
  "comparative_evaluation_executed": false
}
Saved files: ['environment.json', 'execution_summary.json', 'jobs.csv', 'lexical_rankings.csv', 'relevance_labels.csv', 'resumes.csv', 'validation_checks.csv']


## Preliminary interpretation and limitations
The direct-wording resume ranks first for each job in the initial run. For J1, irrelevant R6 ranks above partially relevant R3. These rankings illustrate a possible lexical limitation, not proof that semantic matching will improve results.

I have not executed semantic matching, selected hybrid weights, or computed comparative ranking metrics. The sample is tiny, labels are author-defined, generic words can distort rankings, and negative qualifications are not reliably interpreted. These synthetic results cannot establish real-world recruitment effectiveness.

## Week 2 plan
I will expand and freeze the dataset and relevance rubric, document a pretrained sentence-embedding model, predeclare the hybrid scoring rule, and implement Precision@K and NDCG with known-example tests. Any tuning will be separated from evaluation cases. I will inspect improvements and failures without changing labels to favor my hypothesis.